# Next-Concept Modeling on TinyStories: a small, fair test

This notebook checks the main claim of **Next-Concept Modeling (NCM)** (Kumbong, Khare et al., Stanford, [arXiv:2610.14162](https://arxiv.org/abs/2610.14162)):
NCM matches or beats **masked diffusion** language models while using far fewer generation steps.

It runs in three parts:

| Part | What it does | Why it matters |
|---|---|---|
| **A. Reproduce** | Downloads the authors' released TinyStories model and scores it with their own evaluation script | Shows the official code and metrics run correctly here, so later numbers are trustworthy |
| **B. Small-scale race** | Trains three models from scratch with the **same backbone size, same data, same training-token budget**: NCM (built from the authors' own model classes), a masked-diffusion model (MDLM-style) and a left-to-right (autoregressive) model | Tests whether NCM's advantage holds at a size anyone can train |
| **C. Results** | Generative perplexity under GPT-2 Large, sample diversity, steps and wall-clock time, charts, coarse-to-fine trajectories for a video | Gives you the numbers, charts and samples for the article |

### How to run
1. **Runtime → Change runtime type → T4 GPU** (any GPU works; A100/L4 are faster).
2. Pick a preset in the **Settings** cell (`quick` ≈ 1–1.5 h on a T4, `article` ≈ 4–6 h with 3 training seeds).
3. **Runtime → Run all.** Training cells print an ETA. Optionally mount Google Drive in Settings so a disconnect doesn't lose your models.
4. At the end, download `ncm_results.zip` (results JSON, charts, samples, trajectories).

**Honesty notes built in:** every number in the final table comes from this run; the reference row (real TinyStories text) is scored the same way; all three generators use temperature 1.0 with no top-k/top-p tricks; sampling seeds and training seeds are reported separately.

## Settings

In [ ]:
PRESET = "quick"          # "quick" (~1-1.5 h on a T4, 1 training seed) or "article" (~4-6 h, 3 training seeds)
SAVE_TO_DRIVE = False     # True: keep checkpoints/results in Google Drive (survives Colab disconnects)

import os
TEST_MODE = os.environ.get("NCM_NOTEBOOK_TEST") == "1"   # used only to smoke-test this notebook on CPU; leave as is
if TEST_MODE:
    PRESET = "test"

PRESETS = {
    #                tokenizer        NCM generator    backbone (same for all three models)   evaluation
    "quick":   dict(tok_steps=3000, tok_batch=256, gen_steps=4000, batch=128, n_embd=384, n_layer=6, n_head=6,
                    train_seeds=[0], sample_seeds=[0, 1, 2], n_samples=256, mdlm_steps=[8, 16, 32, 64, 128],
                    n_stories=300_000, lr=5e-4),
    "article": dict(tok_steps=6000, tok_batch=256, gen_steps=10000, batch=128, n_embd=512, n_layer=8, n_head=8,
                    train_seeds=[0, 1, 2], sample_seeds=[0, 1, 2, 3, 4], n_samples=256, mdlm_steps=[8, 12, 16, 32, 64, 128],
                    n_stories=1_000_000, lr=5e-4),
    "test":    dict(tok_steps=30, tok_batch=16, gen_steps=30, batch=8, n_embd=64, n_layer=2, n_head=2,
                    train_seeds=[0], sample_seeds=[0], n_samples=8, mdlm_steps=[4, 8],
                    n_stories=400, lr=1e-3),
}
CFG = PRESETS[PRESET]
if TEST_MODE and os.environ.get("NCM_TEST_GEN_STEPS"):
    CFG["gen_steps"] = int(os.environ["NCM_TEST_GEN_STEPS"])
# Same training-token budget for everyone: the baselines get as many sequences as NCM's tokenizer + generator combined.
CFG["baseline_steps"] = (CFG["tok_steps"] * CFG["tok_batch"] + CFG["gen_steps"] * CFG["batch"]) // CFG["batch"]
L = 128   # tokens per sample, as in the paper's TinyStories setup
print(PRESET, CFG)

## Setup: GPU check, the authors' code, dependencies

In [ ]:
import sys, subprocess, time, json, math, random, pickle, shutil, contextlib
from collections import Counter

def sh(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)

IN_COLAB = os.path.isdir("/content")
if SAVE_TO_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/ncm_experiment"
else:
    WORK = "/content/ncm_experiment" if IN_COLAB else os.path.abspath("ncm_experiment")
os.makedirs(WORK, exist_ok=True)
OUT = os.path.join(WORK, f"results_{PRESET}"); os.makedirs(OUT, exist_ok=True)

REPO_COMMIT = "c70274273f489abc73832b2b98584a8483b6b203"   # HazyResearch/ncm, 14 Oct 2026
REPO = os.path.join(WORK, "ncm")
if not os.path.isdir(REPO):
    if TEST_MODE:
        shutil.copytree(os.environ["NCM_REPO_LOCAL"], REPO)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/HazyResearch/ncm.git", REPO], check=True)
        subprocess.run(["git", "checkout", "-q", REPO_COMMIT], cwd=REPO, check=True)
if not TEST_MODE:   # installs into the same Python this notebook runs on (works on Colab, Linux, macOS, Windows)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "omegaconf", "hydra-core", "regex", "rouge_score",
                    "sacrebleu", "huggingface_hub[cli]", "datasets", "transformers", "pandas", "matplotlib", "tabulate"], check=True)
sys.path.insert(0, REPO)
os.environ["NCM_ROOT"] = REPO

import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from omegaconf import OmegaConf
from models.vqvae import VQVAE                       # authors' tokenizer
from models.next_concept import NextConceptTransformer  # authors' generator
from models.common import Block, precompute_rope_cos_sin  # authors' transformer block (baselines reuse it)
from tokenization.bpe_tokenizer import BPETokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    print("No GPU found." + ("" if TEST_MODE else " Switch the runtime to a GPU (Runtime -> Change runtime type), or this will take days."))
    AMP_DTYPE = None

def amp():
    return torch.autocast("cuda", dtype=AMP_DTYPE) if AMP_DTYPE is not None else contextlib.nullcontext()

GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == "cuda" else 0.0

def micro_batch(per_sample_mb, B):
    # Split each batch into chunks that fit in GPU memory; gradients are summed, so the update is the same as one full batch.
    if TEST_MODE: return max(1, B // 2)       # exercises the chunked path in the CPU smoke test
    if DEVICE != "cuda": return B
    m = B
    while m > 1 and m * per_sample_mb > 0.35 * GPU_GB * 1000: m //= 2
    return m

def make_scaler():
    return torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))

def lr_at(step, total, peak, warmup=500, floor_frac=0.1):
    warm = min(1.0, (step + 1) / min(warmup, max(total // 10, 1)))
    return peak * warm * (floor_frac + (1 - floor_frac) * 0.5 * (1 + math.cos(math.pi * min(step / total, 1.0))))

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

def sync():
    if DEVICE == "cuda": torch.cuda.synchronize()

class ETA:
    def __init__(self, name, total): self.name, self.total, self.t0 = name, total, time.time()
    def log(self, step, **kv):
        el = time.time() - self.t0; rem = el / max(step, 1) * (self.total - step)
        vals = " ".join(f"{k} {v:.3f}" for k, v in kv.items())
        print(f"[{self.name}] step {step}/{self.total} {vals} | {el/60:.1f} min elapsed, ~{rem/60:.1f} min left", flush=True)

## Part A — Reproduce the authors' TinyStories model

Downloads the released tokenizer + generator and runs the authors' own `evaluate.py` (5 seeds × 256 samples, generative perplexity under GPT-2 Large).
The paper reports **Gen. PPL 15.41, entropy 3.98** with 12 steps (real TinyStories text: 9.67 / 3.96).
If your numbers land close to that, the official pipeline works here. This part also supplies the 8,192-token BPE vocabulary used in Part B.

In [ ]:
CK_OFFICIAL = os.path.join(WORK, "artifacts", "tinystories")
OFFICIAL_CFG = os.path.join(REPO, "config", "eval", "tinystories.yaml")

if TEST_MODE:
    # byte-level stand-in vocabulary (the real one comes from the authors' checkpoint)
    os.makedirs(CK_OFFICIAL, exist_ok=True)
    vocab = {0: b"<|endoftext|>", 1: b"<|pad|>", **{i + 2: bytes([i]) for i in range(256)}}
    pickle.dump(vocab, open(os.path.join(CK_OFFICIAL, "bpe_vocab_8192.pickle"), "wb"))
    pickle.dump([], open(os.path.join(CK_OFFICIAL, "bpe_merges_8192.pickle"), "wb"))
    OFFICIAL = {"skipped": "test mode"}
elif os.path.exists(os.path.join(OUT, "official_reproduction.json")) and \
        "gen_ppl" in json.load(open(os.path.join(OUT, "official_reproduction.json"))):
    OFFICIAL = json.load(open(os.path.join(OUT, "official_reproduction.json")))
    print("Reusing finished reproduction:", OFFICIAL)
else:
    from huggingface_hub import snapshot_download
    snapshot_download("hazyresearch/ncm-checkpoints", allow_patterns=["tinystories/*"], local_dir=os.path.join(WORK, "artifacts"))
    t0 = time.time()
    res = subprocess.run([sys.executable, "evaluate.py", "--config", OFFICIAL_CFG, "--checkpoint-dir", CK_OFFICIAL,
                          "--out", os.path.join(WORK, "eval_official")], cwd=REPO, capture_output=True, text=True,
                         env=dict(os.environ, NCM_ROOT=REPO))
    print(res.stdout[-3000:]); print(res.stderr[-3000:])
    mean_line = [l for l in res.stdout.splitlines() if l.strip().startswith("mean")]
    OFFICIAL = {"raw_mean_line": mean_line[-1] if mean_line else None, "eval_minutes": (time.time() - t0) / 60}
    if mean_line:
        parts = mean_line[-1].split()
        OFFICIAL.update(gen_ppl=float(parts[1]), entropy=float(parts[2]))
    print("Official TinyStories model:", OFFICIAL, " | paper: Gen PPL 15.41, entropy 3.98")
json.dump(OFFICIAL, open(os.path.join(OUT, "official_reproduction.json"), "w"), indent=1)

Coarse-to-fine trajectories from the **official** model (great material for a video): each row decodes the text after one more level.

In [ ]:
def load_official():
    cfg = OmegaConf.load(OFFICIAL_CFG)
    cfg.tokenizer.pre_tokenizer.vocab_file_path = os.path.join(CK_OFFICIAL, cfg.checkpoints.bpe_vocab)
    cfg.tokenizer.pre_tokenizer.merges_file_path = os.path.join(CK_OFFICIAL, cfg.checkpoints.bpe_merges)
    vq = VQVAE(**{k: v for k, v in cfg.tokenizer.vqvae.items() if k != "checkpoint_path"})
    vq.load_state_dict(torch.load(os.path.join(CK_OFFICIAL, cfg.checkpoints.tokenizer), map_location="cpu", weights_only=False)["model"])
    vq = vq.to(DEVICE).eval()
    net = NextConceptTransformer(vqvae_local=vq, config=cfg.model).to(DEVICE)
    sd = torch.load(os.path.join(CK_OFFICIAL, cfg.checkpoints.generator), map_location="cpu", weights_only=False)
    net.load_state_dict(sd["model"]); net.eval()
    l0p = sd["inference"]["level0_unigram"].float()
    return vq, net, l0p

def cumulative_zhats(vq, idx):
    # Partial reconstructions after each level. (The repo's idx_to_zhats adds in place, so its list repeats the final level.)
    q, levels = vq.vq, vq.vq.levels
    z, out, off = torch.zeros(idx.shape[0], levels[-1], vq.embed_dim, device=idx.device), [], 0
    for k, l in enumerate(levels):
        e = q.embeddings[k](idx[:, off:off + l])
        e = q._up_to_full(e, k, levels[-1]) if k != len(levels) - 1 else e
        z = z + q._apply_quant_resi(e, k); out.append(z); off += l
    return out

def trajectories(vq, net, l0p, bpe, n=3, seed=0):
    torch.manual_seed(seed)
    l0 = torch.multinomial(l0p.cpu(), n * vq.vq.levels[0], replacement=True).view(n, -1).to(DEVICE)
    eos = bpe.eos_token_id
    with torch.no_grad():
        g = net.generate(l0, torch.full((n, 1), eos, device=DEVICE), temperature=1.0, top_k=0, top_p=1.0)
        zhats = cumulative_zhats(vq, g)
        rows = [[bpe.decode([t for t in vq.decoder(z).argmax(-1)[i].tolist() if t not in (eos, bpe.pad_token_id)])
                 for z in zhats] for i in range(n)]
    return rows

if not TEST_MODE:
    vq_o, net_o, l0p_o = load_official()
    bpe_o = BPETokenizer.from_config(OmegaConf.create({"vocab_file_path": os.path.join(CK_OFFICIAL, "bpe_vocab_8192.pickle"),
            "merges_file_path": os.path.join(CK_OFFICIAL, "bpe_merges_8192.pickle"),
            "special_tokens": {"eos_token": "<|endoftext|>", "pad_token": "<|pad|>"}}))
    TRAJ_OFFICIAL = trajectories(vq_o, net_o, l0p_o, bpe_o)
    for k, s in enumerate(TRAJ_OFFICIAL[0]):
        print(f"level {k+1:2d} ({vq_o.vq.levels[k]:3d} codes): {s[:160]!r}")
    json.dump(TRAJ_OFFICIAL, open(os.path.join(OUT, "trajectories_official.json"), "w"), indent=1)
    del vq_o, net_o
    if DEVICE == "cuda": torch.cuda.empty_cache()

## Part B — Data: TinyStories with the authors' 8,192-token BPE

Each training example is the first 128 tokens of one story (story + end-of-text, padded), matching the paper's setup.
A cached re-implementation of the authors' BPE encoder is checked token-for-token against theirs before use.

In [ ]:
SPECIAL = {"eos_token": "<|endoftext|>", "pad_token": "<|pad|>"}
BPE = BPETokenizer.from_config(OmegaConf.create({"vocab_file_path": os.path.join(CK_OFFICIAL, "bpe_vocab_8192.pickle"),
        "merges_file_path": os.path.join(CK_OFFICIAL, "bpe_merges_8192.pickle"), "special_tokens": SPECIAL}))
EOS, PAD = BPE.eos_token_id, BPE.pad_token_id
VOCAB = max(BPE.vocab.keys()) + 1
print("vocab", VOCAB, "eos", EOS, "pad", PAD)

import regex as re
_PAT = re.compile(r"'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+")
_cache = {}
def bpe_word(w):
    if w in _cache: return _cache[w]
    ids = BPE.encode(w)          # authors' encoder, applied once per distinct pre-token
    _cache[w] = ids; return ids
def fast_encode(text):
    out = []
    for w in _PAT.findall(text): out += bpe_word(w)
    return out

def to_example(text):
    ids = (fast_encode(text) + [EOS])[:L]
    return ids + [PAD] * (L - len(ids))

if TEST_MODE:
    raw = open(os.environ["NCM_TEST_TEXT"]).read().split("\n\n")
    train_texts, val_texts = raw[:CFG["n_stories"]], raw[-200:]
else:
    from datasets import load_dataset
    train_texts = load_dataset("roneneldan/TinyStories", split=f"train[:{CFG['n_stories']}]")["text"]
    val_texts = load_dataset("roneneldan/TinyStories", split="validation[:2000]")["text"]

for t in train_texts[:200]:                     # fast encoder must equal the authors' encoder
    assert fast_encode(t) == BPE.encode(t), "fast BPE mismatch"
t0 = time.time()
def encode_all(texts):     # fills a compact int16 array row by row (keeps RAM low on 16 GB machines)
    arr = np.empty((len(texts), L), dtype=np.int16)
    for i, t in enumerate(texts):
        arr[i] = to_example(t)
        if (i + 1) % 100_000 == 0: print(f"  tokenized {i+1:,}/{len(texts):,}", flush=True)
    return torch.from_numpy(arr)
DATA, VAL = encode_all(train_texts), encode_all(val_texts)   # 8,192-token vocab fits in int16
del train_texts
print(f"train {tuple(DATA.shape)}, val {tuple(VAL.shape)}, tokenized in {time.time()-t0:.0f}s")
print("example:", BPE.decode([t for t in DATA[0].tolist() if t not in (EOS, PAD)])[:200])

def batch(data, B, g=None):
    return data[torch.randint(len(data), (B,), generator=g)].long().to(DEVICE)

### B1. NCM tokenizer (the authors' multiscale residual VQ-VAE)

Same architecture and settings as the authors' TinyStories tokenizer config (12 levels: 1, 8, 16, … 128 codes; 4,096-entry codebooks updated by moving averages; fine-level dropout 0.5),
trained here with fewer steps. The training loop (cross-entropy + VQ loss, AdamW, cosine schedule) follows the paper's Eq. 9 and Appendix C.

In [ ]:
VQ_CFG = OmegaConf.load(OFFICIAL_CFG).tokenizer.vqvae
VQ_KW = {k: v for k, v in VQ_CFG.items() if k != "checkpoint_path"}
VQ_KW["in_vocab_size"] = VOCAB
if TEST_MODE:
    VQ_KW.update(embed_dim=64, vqvae_vocab_size=[64] * 12, n_layers=2)
LEVELS = list(VQ_KW["vector_quantizer_config"]["levels"]["manual_levels"])
K = len(LEVELS)

@torch.no_grad()
def recon_by_level(vq, data, n=1024):
    g = torch.Generator().manual_seed(0); x = batch(data, n, g)
    z = vq._encode_pre_quant(x)
    _, zhats = vq.vq.z_to_idx(z, return_zhats=True)
    return [(vq.decoder(zh).argmax(-1) == x).float().mean().item() for zh in zhats]

def train_tokenizer():
    path = os.path.join(OUT, "tokenizer.pt")
    vq = VQVAE(**VQ_KW).to(DEVICE)
    if os.path.exists(path):
        vq.load_state_dict(torch.load(path, map_location=DEVICE)); print("loaded", path); return vq.eval()
    seed_all(0)
    opt = torch.optim.AdamW([p for p in vq.parameters() if p.requires_grad], lr=CFG["lr"], betas=(0.9, 0.95), weight_decay=0.01)
    eta = ETA("tokenizer", CFG["tok_steps"])
    for step in range(1, CFG["tok_steps"] + 1):
        for gr in opt.param_groups: gr["lr"] = lr_at(step, CFG["tok_steps"], CFG["lr"])
        x = batch(DATA, CFG["tok_batch"])
        logits, vq_loss, *_ = vq(x)                    # fp32: nearest-code search is precision-sensitive
        ce = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), x.reshape(-1))
        loss = ce + vq_loss
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(vq.parameters(), 1.0); opt.step()
        if step % max(CFG["tok_steps"] // 10, 1) == 0: eta.log(step, ce=ce.item(), vq=float(vq_loss))
    torch.save(vq.state_dict(), path)
    return vq.eval()

VQ = train_tokenizer()
for p in VQ.parameters(): p.requires_grad_(False)
RECON = recon_by_level(VQ, VAL)
print("exact token reconstruction after each level:", [f"{a:.1%}" for a in RECON])

with torch.no_grad():                                  # level-0 code frequencies (sampled, not predicted, as in the paper)
    cnt = torch.zeros(VQ.vq.vqvae_vocab_sizes[0], device=DEVICE)
    for i in range(0, min(len(DATA), 50_000), 512):
        idx = VQ.encode(DATA[i:i + 512].long().to(DEVICE))[:, :LEVELS[0]]
        cnt += torch.bincount(idx.flatten(), minlength=cnt.numel()).float()
    L0_PROBS = (cnt / cnt.sum()).cpu()
json.dump({"recon_by_level": RECON, "levels": LEVELS}, open(os.path.join(OUT, "tokenizer_report.json"), "w"), indent=1)

### B2. NCM generator (the authors' `NextConceptTransformer`)

Block-diagonal attention, codebook-tied output heads, RoPE per level, a single end-of-text prefix token — the authors' TinyStories generator config, scaled down to the shared backbone size.
Training follows Algorithm 1: encode → corrupt the conditioning codes (linearly decreasing rate, mean 0.5) → predict every level's clean codes in one forward pass.

In [ ]:
def ncm_config():
    c = OmegaConf.load(OFFICIAL_CFG).model
    c.n_embd, c.n_layer, c.n_head = CFG["n_embd"], CFG["n_layer"], CFG["n_head"]
    c.vocab_size = list(VQ.vq.vqvae_vocab_sizes[1:])
    c.prefix_pad_token_id = PAD
    return c

CORRUPT = torch.linspace(0.8, 0.2, K - 1)               # rates for the K-1 conditioning levels, mean 0.5

def corrupt_codes(idx):
    idx = idx.clone(); off = 0
    for k in range(K - 1):
        sl = idx[:, off:off + LEVELS[k]]
        m = torch.rand(sl.shape, device=idx.device) < CORRUPT[k]
        sl[m] = torch.randint(VQ.vq.vqvae_vocab_sizes[k], (int(m.sum()),), device=idx.device)
        off += LEVELS[k]
    return idx

def train_ncm(seed):
    path = os.path.join(OUT, f"ncm_seed{seed}.pt")
    seed_all(seed)
    net = NextConceptTransformer(vqvae_local=VQ, config=ncm_config()).to(DEVICE)
    if os.path.exists(path):
        net.load_state_dict(torch.load(path, map_location=DEVICE)); print("loaded", path)
        logp = path.replace(".pt", "_log.json")
        return net.eval(), (json.load(open(logp)) if os.path.exists(logp) else None)
    if AMP_DTYPE is not None:   # keep the block-diagonal mask in the same precision as attention under autocast
        net.attn_mask_for_masking = net.attn_mask_for_masking.to(AMP_DTYPE)
    opt = torch.optim.AdamW(net.parameters(), lr=3e-4 * (2 if TEST_MODE else 1), betas=(0.9, 0.95), weight_decay=0.1)
    scaler, eta, log, t0 = make_scaler(), ETA(f"NCM seed {seed}", CFG["gen_steps"]), [], time.time()
    S = sum(LEVELS)   # positions per example: every level is trained in one pass (641 vs 128 for the baselines)
    per_sample_mb = (CFG["n_layer"] * CFG["n_head"] * S * S * 8 + CFG["n_layer"] * S * CFG["n_embd"] * 40) / 1e6
    mb = micro_batch(per_sample_mb, CFG["batch"]); n_mb = CFG["batch"] // mb
    print(f"NCM: batch {CFG['batch']} as {n_mb} chunk(s) of {mb} (GPU {GPU_GB:.0f} GB)")
    if DEVICE == "cuda": torch.cuda.empty_cache()
    net.train()
    for step in range(1, CFG["gen_steps"] + 1):
        for gr in opt.param_groups: gr["lr"] = lr_at(step, CFG["gen_steps"], opt.defaults["lr"])
        x = batch(DATA, CFG["batch"])
        opt.zero_grad(set_to_none=True); total = 0.0
        for j in range(n_mb):
            xs = x[j * mb:(j + 1) * mb]
            with torch.no_grad():
                idx = VQ.encode(xs)                                    # all levels' codes
                target = idx[:, LEVELS[0]:]
                cond = VQ.vq.idx_to_ncm_input(corrupt_codes(idx))     # cumulative reconstructions, resized per level
                prefix = VQ.encoder.wte(torch.full((xs.shape[0], 1), EOS, device=DEVICE))
                inp = torch.cat([prefix, cond], 1)
            with amp():
                logits = net(inp)
                loss = F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]), target.reshape(-1)) / n_mb
            scaler.scale(loss).backward(); total += loss.item()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0); scaler.step(opt); scaler.update()
        if step % max(CFG["gen_steps"] // 10, 1) == 0:
            eta.log(step, loss=total); log.append((step, total))
    torch.save(net.state_dict(), path)
    info = {"log": log, "train_minutes": (time.time() - t0) / 60}
    json.dump(info, open(path.replace(".pt", "_log.json"), "w"))
    return net.eval(), info

@torch.no_grad()
def sample_ncm(net, n, temp=1.0):
    l0 = torch.multinomial(L0_PROBS, n * LEVELS[0], replacement=True).view(n, LEVELS[0]).to(DEVICE)
    g = net.generate(l0, torch.full((n, 1), EOS, device=DEVICE), temperature=temp, top_k=0, top_p=1.0)
    zhats, _ = VQ.vq.idx_to_zhats(g, return_zlevels=True)
    return VQ.decoder(zhats[-1]).argmax(-1)                       # same decoding as the authors' eval script

### B3. Baselines on the same backbone

Both reuse the authors' transformer `Block` (QK-norm, RoPE) at the same width and depth as the NCM generator.

* **Masked diffusion (MDLM-style):** linear masking schedule, 1/t-weighted loss on masked tokens; ancestral sampling with T steps (T = number of function evaluations); float64 categorical sampling to avoid the known float32 low-diversity artefact.
* **Autoregressive:** next-token prediction with a KV cache; 128 steps per sample.

Each baseline trains on as many sequences as NCM's tokenizer and generator **combined** (same training-token budget, the paper's accounting).

In [ ]:
class Backbone(nn.Module):
    def __init__(self, vocab_in, vocab_out, causal):
        super().__init__()
        c = type("C", (), dict(n_embd=CFG["n_embd"], n_head=CFG["n_head"], bias=True, dropout=0.1, use_qk_norm=True))
        self.causal = causal
        self.wte = nn.Embedding(vocab_in, CFG["n_embd"])
        self.blocks = nn.ModuleList([Block(c) for _ in range(CFG["n_layer"])])
        self.ln = nn.LayerNorm(CFG["n_embd"]); self.head = nn.Linear(CFG["n_embd"], vocab_out, bias=False)
        cos, sin = precompute_rope_cos_sin(torch.arange(L + 1), CFG["n_embd"] // CFG["n_head"], 10000)
        self.register_buffer("cos", cos, persistent=False); self.register_buffer("sin", sin, persistent=False)
        self.apply(self._init)
    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)): nn.init.normal_(m.weight, 0.0, 0.02)
        if isinstance(m, nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
    def forward(self, x):
        T = x.shape[1]
        mask = None if self.causal else torch.ones(T, T, dtype=torch.bool, device=x.device)
        h = self.wte(x)
        for b in self.blocks: h = b(h, attn_mask=mask, rope_cos_sin=(self.cos, self.sin))
        return self.head(self.ln(h))

def train_baseline(kind, seed):
    path = os.path.join(OUT, f"{kind}_seed{seed}.pt")
    seed_all(seed)
    model = (Backbone(VOCAB + 1, VOCAB, causal=False) if kind == "mdlm" else Backbone(VOCAB, VOCAB, causal=True)).to(DEVICE)
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=DEVICE)); print("loaded", path)
        logp = path.replace(".pt", "_log.json")
        return model.eval(), (json.load(open(logp)) if os.path.exists(logp) else None)
    MASK = VOCAB
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4 * (2 if TEST_MODE else 1), betas=(0.9, 0.95), weight_decay=0.1)
    steps = CFG["baseline_steps"]
    scaler, eta, log, t0 = make_scaler(), ETA(f"{kind} seed {seed}", steps), [], time.time()
    per_sample_mb = (CFG["n_layer"] * CFG["n_head"] * L * L * 8 + CFG["n_layer"] * L * CFG["n_embd"] * 40 + L * VOCAB * 12) / 1e6
    mb = micro_batch(per_sample_mb, CFG["batch"]); n_mb = CFG["batch"] // mb
    print(f"{kind}: batch {CFG['batch']} as {n_mb} chunk(s) of {mb}")
    if DEVICE == "cuda": torch.cuda.empty_cache()
    model.train()
    for step in range(1, steps + 1):
        for gr in opt.param_groups: gr["lr"] = lr_at(step, steps, opt.defaults["lr"])
        x_full = batch(DATA, CFG["batch"])
        B_full = x_full.shape[0]
        t_full = ((torch.rand(1, device=DEVICE) + torch.arange(B_full, device=DEVICE) / B_full) % 1).clamp(min=1e-3)[:, None]  # antithetic t
        opt.zero_grad(set_to_none=True); total = 0.0
        for j in range(n_mb):
            x, t = x_full[j * mb:(j + 1) * mb], t_full[j * mb:(j + 1) * mb]
            with amp():
                if kind == "ar":
                    inp = torch.cat([torch.full((x.shape[0], 1), EOS, device=DEVICE), x[:, :-1]], 1)
                    loss = F.cross_entropy(model(inp).float().reshape(-1, VOCAB), x.reshape(-1))
                else:
                    masked = torch.rand(x.shape, device=DEVICE) < t
                    logits = model(torch.where(masked, torch.full_like(x, MASK), x)).float()
                    nll = F.cross_entropy(logits.reshape(-1, VOCAB), x.reshape(-1), reduction="none").view_as(x)
                    loss = (nll * masked / t).sum() / x.numel()
                loss = loss / n_mb
            scaler.scale(loss).backward(); total += loss.item()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update()
        if step % max(steps // 10, 1) == 0:
            eta.log(step, loss=total); log.append((step, total))
    torch.save(model.state_dict(), path)
    info = {"log": log, "train_minutes": (time.time() - t0) / 60}
    json.dump(info, open(path.replace(".pt", "_log.json"), "w"))
    return model.eval(), info

@torch.no_grad()
def sample_mdlm(model, n, T):
    MASK = VOCAB
    x = torch.full((n, L), MASK, device=DEVICE)
    ts = torch.linspace(1, 0, T + 1, device=DEVICE)
    for i in range(T):
        t, s = ts[i], ts[i + 1]
        logits = model(x)
        is_m = x == MASK
        unmask = is_m & (torch.rand(x.shape, device=DEVICE, dtype=torch.double) < (t - s) / t)
        if i == T - 1: unmask = is_m
        pos = unmask.nonzero(as_tuple=True)
        if pos[0].numel():   # draw only where tokens are revealed this step (same distribution, far less float64 work)
            x[pos] = torch.multinomial(F.softmax(logits[pos].double(), -1), 1).squeeze(-1)
    return x

@torch.no_grad()
def sample_ar(model, n, temp=1.0):
    for b in model.blocks: b.attn.kv_caching(True)
    try:
        cur, out = torch.full((n, 1), EOS, device=DEVICE), []
        for t in range(L):
            h = model.wte(cur)
            cs = (model.cos[: t + 1], model.sin[: t + 1])
            for b in model.blocks: h = b(h, attn_mask=None, rope_cos_sin=cs)
            p = F.softmax(model.head(model.ln(h[:, -1])).double() / temp, -1)
            cur = torch.multinomial(p, 1); out.append(cur)
        return torch.cat(out, 1)
    finally:
        for b in model.blocks: b.attn.kv_caching(False)

### B4. Train everything (prints an ETA per model)

In [ ]:
MODELS, TRAIN_INFO = {}, {}
for s in CFG["train_seeds"]:
    MODELS[("ncm", s)], TRAIN_INFO[f"ncm_seed{s}"] = train_ncm(s)
    MODELS[("mdlm", s)], TRAIN_INFO[f"mdlm_seed{s}"] = train_baseline("mdlm", s)
    MODELS[("ar", s)], TRAIN_INFO[f"ar_seed{s}"] = train_baseline("ar", s)
def nparams(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
PARAMS = {k: nparams(MODELS[(k, CFG["train_seeds"][0])]) for k in ("ncm", "mdlm", "ar")}
print("trainable parameters:", PARAMS)
json.dump({"train": TRAIN_INFO, "params": PARAMS, "cfg": CFG}, open(os.path.join(OUT, "training.json"), "w"), indent=1)

## Part C — Evaluation

* **Gen. PPL**: perplexity of the samples under GPT-2 Large, using the authors' `generation_perplexity` function (lower = more fluent). Like the paper, it is reported **together with unigram entropy**, because a model can lower Gen. PPL by becoming repetitive.
* **Entropy**: mean per-sample unigram entropy of BPE tokens (real text ≈ 3.96 in the paper).
* **Steps (NFE)**: forward passes per sample. **Time**: wall-clock seconds for 256 samples at batch 64, fp32, same GPU.

In [ ]:
def to_text(ids):
    ids = list(ids)
    if EOS in ids: ids = ids[:ids.index(EOS)]
    return BPE.decode([t for t in ids if t not in (EOS, PAD)], skip_special_tokens=True)

def entropy(ids):
    c, m = Counter(ids), len(ids)
    return -sum(v / m * math.log(v / m) for v in c.values()) if m else 0.0

if TEST_MODE:
    def gen_ppl(texts):            # stand-in scorer for the CPU smoke test only
        return float(np.exp(np.mean([8 - min(len(set(t)), 8) for t in texts] or [0])))
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from evaluation.gen_ppl import generation_perplexity
    REF_TOK = AutoTokenizer.from_pretrained("gpt2-large")
    if DEVICE == "cuda": torch.cuda.empty_cache()
    REF_LM = AutoModelForCausalLM.from_pretrained("gpt2-large").to(DEVICE).eval()
    SCORE_BS = 32 if GPU_GB > 12 else 8
    def gen_ppl(texts):
        return generation_perplexity(REF_LM, REF_TOK, [""] * len(texts), texts, device=DEVICE, batch_size=SCORE_BS).mean_ppl

def score(texts):
    return {"gen_ppl": gen_ppl(texts), "entropy": float(np.mean([entropy(BPE.encode(t)) for t in texts if t]))}

def run_sampler(fn, n, bs=64):
    outs = []; sync(); t0 = time.time()
    for i in range(0, n, bs): outs.append(fn(min(bs, n - i)).cpu())
    sync(); return torch.cat(outs), time.time() - t0

RESULTS, SAMPLES = [], {}
ref_texts = [to_text(x) for x in VAL[: CFG["n_samples"]].tolist()]
RESULTS.append({"model": "Real TinyStories text", "nfe": None, "train_seed": None, **score(ref_texts)})
SAMPLES["data"] = ref_texts[:5]

configs = [("ncm", f"NCM ({K - 1} levels predicted)", K - 1, lambda m: (lambda n: sample_ncm(m, n)))]
configs += [("mdlm", f"Masked diffusion, T={T}", T, (lambda T: lambda m: (lambda n: sample_mdlm(m, n, T)))(T)) for T in CFG["mdlm_steps"]]
configs += [("ar", "Autoregressive", L, lambda m: (lambda n: sample_ar(m, n)))]

for kind, name, nfe, make in configs:
    for ts in CFG["train_seeds"]:
        model = MODELS[(kind, ts)]
        for ss in CFG["sample_seeds"]:
            seed_all(1000 + ss)
            ids, sec = run_sampler(make(model), CFG["n_samples"])
            texts = [to_text(r) for r in ids.tolist()]
            row = {"model": name, "kind": kind, "nfe": nfe, "train_seed": ts, "sample_seed": ss,
                   "seconds_per_256": sec * 256 / CFG["n_samples"], **score(texts)}
            RESULTS.append(row); print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in row.items()})
            SAMPLES.setdefault(name, texts[:5]) if ss == CFG["sample_seeds"][0] and ts == CFG["train_seeds"][0] else None

json.dump({"results": RESULTS, "samples": SAMPLES, "official": OFFICIAL, "recon_by_level": RECON, "cfg": CFG},
          open(os.path.join(OUT, "results.json"), "w"), indent=1)

### Summary table (mean ± std over all training × sampling seeds) — paste this into your article

In [ ]:
import pandas as pd
df = pd.DataFrame(RESULTS)
g = df.groupby("model", sort=False).agg(steps=("nfe", "first"), gen_ppl=("gen_ppl", "mean"), gen_ppl_sd=("gen_ppl", "std"),
                                         entropy=("entropy", "mean"), seconds_per_256=("seconds_per_256", "mean")).reset_index()
g["Gen. PPL ↓"] = [f"{m:.2f}" + ("" if pd.isna(s) else f" ± {s:.2f}") for m, s in zip(g.gen_ppl, g.gen_ppl_sd)]
g["Entropy"] = g.entropy.map(lambda v: f"{v:.2f}")
g["Steps"] = g.steps.map(lambda v: "–" if pd.isna(v) else str(int(v)))
g["Seconds / 256 samples"] = g.seconds_per_256.map(lambda v: "–" if pd.isna(v) else f"{v:.1f}")
TABLE = g[["model", "Steps", "Gen. PPL ↓", "Entropy", "Seconds / 256 samples"]].rename(columns={"model": "Model"})
if isinstance(OFFICIAL, dict) and "gen_ppl" in OFFICIAL:
    print(f"Official released NCM (~89M params, full training): Gen PPL {OFFICIAL['gen_ppl']:.2f}, entropy {OFFICIAL['entropy']:.2f} (paper: 15.41 / 3.98)\n")
print(f"Backbone: {CFG['n_layer']} layers x {CFG['n_embd']} wide; trainable params {PARAMS}; training seeds {CFG['train_seeds']}, sampling seeds {CFG['sample_seeds']}\n")
try:
    md_table = TABLE.to_markdown(index=False)      # needs the 'tabulate' package (preinstalled on Colab)
except ImportError:
    md_table = TABLE.to_string(index=False)
print(md_table)
open(os.path.join(OUT, "summary_table.md"), "w").write(md_table)
TABLE

### Charts

In [ ]:
import matplotlib; import matplotlib.pyplot as plt
INK, INK2, BG, GRID = "#0b0b0b", "#52514e", "#fcfcfb", "#ebeae6"
COL = {"mdlm": "#2a78d6", "ncm": "#eb6834", "ar": "#1baf7a", "data": "#9b9a94"}
plt.rcParams.update({"figure.facecolor": BG, "axes.facecolor": BG, "axes.edgecolor": "#d6d5d0", "axes.labelcolor": INK2,
                     "xtick.color": INK2, "ytick.color": INK2, "axes.spines.top": False, "axes.spines.right": False, "font.size": 12})
agg = df[df.kind.notna()].groupby(["kind", "model"], sort=False).agg(nfe=("nfe", "first"), ppl=("gen_ppl", "mean"),
      sd=("gen_ppl", "std"), ent=("entropy", "mean"), sec=("seconds_per_256", "mean")).reset_index().fillna({"sd": 0})
data_ppl = df[df.model == "Real TinyStories text"].gen_ppl.iloc[0]

def panel(ax, xcol, xlabel, logx=True):
    for kind, label in [("mdlm", "Masked diffusion"), ("ncm", "NCM"), ("ar", "Autoregressive")]:
        d = agg[agg.kind == kind].sort_values(xcol)
        ax.errorbar(d[xcol], d.ppl, yerr=d.sd, color=COL[kind], marker="o", ms=8, lw=2, capsize=3, label=label,
                    markeredgecolor=BG, markeredgewidth=1.5)
        for _, r in d.iterrows():
            ax.annotate(f"H={r.ent:.2f}", (r[xcol], r.ppl), textcoords="offset points", xytext=(6, 6), fontsize=9, color=INK2)
    ax.axhline(data_ppl, color=COL["data"], ls="--", lw=1.5, label=f"Real text ({data_ppl:.1f})")
    if logx:
        ax.set_xscale("log")
        ax.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:g}"))
        ax.xaxis.set_major_locator(matplotlib.ticker.LogLocator(subs=(1, 2, 5)))
        ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
    ax.set_xlabel(xlabel); ax.set_ylabel("Gen. PPL under GPT-2 Large (lower is better)")
    ax.yaxis.grid(True, color=GRID); ax.set_axisbelow(True); ax.legend(frameon=False, loc="best")

fig, axs = plt.subplots(1, 2, figsize=(14, 5.2))
panel(axs[0], "nfe", "Generation steps per sample (NFE, log scale)")
panel(axs[1], "sec", "Seconds to generate 256 samples (log scale)")
fig.suptitle(f"TinyStories, 128 tokens, same {CFG['n_layer']}x{CFG['n_embd']} backbone and training-token budget  (H = sample entropy; real text ≈ "
             f"{df[df.model=='Real TinyStories text'].entropy.iloc[0]:.2f})", x=0.01, ha="left", color=INK, fontweight="bold")
fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig_quality_vs_steps_and_time.png"), dpi=160); plt.show()

fig, ax = plt.subplots(figsize=(8, 4.4))
ax.plot(range(1, K + 1), RECON, color=COL["ncm"], marker="o", lw=2, ms=8, markeredgecolor=BG)
ax.set_xticks(range(1, K + 1)); ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0)); ax.yaxis.grid(True, color=GRID)
ax.set_xlabel("Levels decoded"); ax.set_ylabel("Tokens reconstructed exactly")
ax.set_title("NCM tokenizer: reconstruction builds up level by level", loc="left", color=INK, fontweight="bold")
fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig_tokenizer_levels.png"), dpi=160); plt.show()

fig, ax = plt.subplots(figsize=(8, 4.4))
for k, info in TRAIN_INFO.items():
    if info and info["log"]:
        s, l = zip(*info["log"]); kind = k.split("_")[0]
        ax.plot(s, l, color=COL[kind], lw=2, label=k)
ax.set_xlabel("Training step"); ax.set_ylabel("Training loss (each model's own objective)")
ax.yaxis.grid(True, color=GRID); ax.legend(frameon=False, fontsize=9)
ax.set_title("Training curves (losses are not comparable across model types)", loc="left", color=INK, fontweight="bold")
fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig_training_curves.png"), dpi=160); plt.show()

### Samples and coarse-to-fine trajectories (for the article and the video)

In [ ]:
for name, texts in SAMPLES.items():
    print(f"\n=== {name} ===")
    for t in texts[:2]: print("  ", t[:300].replace("\n", " "))

TRAJ_OURS = trajectories(VQ, MODELS[("ncm", CFG["train_seeds"][0])], L0_PROBS, BPE)
print("\n=== Our small NCM, decoded after each level ===")
for k, s in enumerate(TRAJ_OURS[0]):
    print(f"level {k+1:2d} ({LEVELS[k]:3d} codes): {s[:160]!r}")
json.dump(TRAJ_OURS, open(os.path.join(OUT, "trajectories_ours.json"), "w"), indent=1)

## Download everything

In [ ]:
import zipfile
zip_path = os.path.join(WORK, "ncm_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:      # results, charts, samples (checkpoints stay in WORK)
    for f in sorted(os.listdir(OUT)):
        if not f.endswith(".pt"): z.write(os.path.join(OUT, f), f)
print("saved", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)")
if IN_COLAB and not TEST_MODE:
    from google.colab import files
    files.download(zip_path)

## Reading the results honestly

* **NCM vs masked diffusion at the same step count** is the paper's central claim; compare the orange point (NCM, 11 predicted levels + 1 sampled) with the blue curve near the same NFE.
* Always read Gen. PPL **with** entropy: a lower Gen. PPL with entropy far below real text means repetitive samples, not better ones.
* Training seeds vary more than sampling seeds; with only one training seed, call differences under ~10% "inconclusive".
* These models are ~20–50× smaller and trained ~50–100× shorter than the paper's; the official row in Part A shows what full scale reaches.
* Cite: Kumbong, Khare, et al. *Next-Concept Modeling: Fast, Parallel Coarse-to-Fine Text Generation.* arXiv:2610.14162 (2026). Code: github.com/HazyResearch/ncm (MIT).